In [ ]:
pip install nixtla

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.6 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
from nixtla import NixtlaClient
from utilsforecast.preprocessing import fill_gaps
from utilsforecast.losses import mae
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error
import warnings

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

In [ ]:
nixtla_client = NixtlaClient(
    api_key="API-KEY"
)

nixtla_client.validate_api_key()

True

In [ ]:
def train_linear(df, ride_name):
  df = df.dropna()

  df.rename(columns={"datetime_idx":"ds"}, inplace=True)
  df["ds"] = pd.to_datetime(df["ds"].astype(str))
  df["unique_id"] = ride_name
  df = fill_gaps(df, freq="min")
  df["wait_time"] = df["wait_time"].interpolate(method="linear", limit_direction="both")

  df.set_index("ds", inplace=True)

  X = df[[
        "wait_time", "unique_id", 'minutes_from_open', 'minutes_to_close',
        'duration', 'min_height', 'has_express', "has_ea",
        'is_kids_ride', 'is_thrill_ride', 'is_dark_ride', 'is_water_ride', 'has_indoor_queue',
        'temp', 'rhum', 'prcp', 'wspd', 'pres', 'cldc', 'coco',
        'is_holiday', 'is_major_holiday', 'is_annual_event',
        'day_of_week_sin', 'day_of_week_cos'
  ]]

  X_train = X.loc[pd.to_datetime("2024-01-01 00:00:00"):pd.to_datetime("2025-01-01 00:00:00")]
  X_test = X.loc[pd.to_datetime("2025-01-01 00:00:00"):]

  X_train.reset_index(inplace=True)
  X_test.reset_index(inplace=True)

  X_test = X_test.iloc[0:186150,:]

  pred = nixtla_client.forecast(
      df=X_train,
      X_df=X_train.drop(columns=["wait_time"]),
      h=186150,
      freq="min",
      finetune_loss="mae",
      time_col="ds",
      target_col="wait_time",
      model="timegpt-1-long-horizon",
      num_partitions=2
  )

  mae = mean_absolute_error(X_test["wait_time"], pred["TimeGPT"])
  rmse = root_mean_squared_error(X_test["wait_time"], pred["TimeGPT"])
  wape = sum(np.abs(np.array(X_test["wait_time"]) - np.array(pred["TimeGPT"]))) / sum(X_test["wait_time"])

  return pred[["ds", "TimeGPT"]].rename(columns={"ds":"datetime_idx", "TimeGPT":"wait_time"}), mae, rmse, wape

def theme_park_linear(folder_path):
  for file_name in os.listdir(f"{folder_path}Wait_Time_Features/"):
    ride_name = file_name.replace("_features.parquet", "")
    print(ride_name)
    try:
      features = pd.read_parquet(f"{folder_path}Wait_Time_Features/{file_name}")
      results = pd.read_csv(f"{folder_path}Results/timeGPT_results.csv")

      predictions, mae, rmse, wape = train_linear(features, ride_name)

      results = pd.concat([results, pd.DataFrame({"ride_name": ride_name, "mae": mae, "rmse": rmse, "wape": wape}, index=[0])], ignore_index=True)

      results.to_csv(f"{folder_path}Results/timeGPT_results.csv", index=False)
      predictions.to_csv(f"{folder_path}Predictions/TimeGPT/{ride_name}_predictions.csv", index=False)
    except Exception as e:
      print(e)
      continue


In [ ]:
# Create Linear Models for Islands of Adventure
theme_park_linear("/content/drive/MyDrive/Theme_Park_Data/Universal_IOA/")

# Create Linear Models for Universal Studios
theme_park_linear("/content/drive/MyDrive/Theme_Park_Data/Universal_Studios_FL/")

caroseussel


  0%|          | 0/1 [00:00<?, ?it/s]

doctor-dooms-fearfall


  0%|          | 0/1 [00:00<?, ?it/s]

the-amazing-adventures-of-spiderman


...

  0%|          | 0/1 [00:00<?, ?it/s]

jurassic-world-velocicoaster


  0%|          | 0/1 [00:00<?, ?it/s]

dudley-dorights-ripsaw-falls


  0%|          | 0/1 [00:00<?, ?it/s]